# 6.3 · Union-find and spanning trees

*Graphs* · 8 exercises

Keep groups that only ever merge: a forest of parent links with path compression and union by size answers same-group questions in near-constant time, which drives cycle checks and minimum spanning trees.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-6.3) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-6.3) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-6.3)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import heapq

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 6.3.1 · Union-find  `Core`

**Task.** Build `DisjointSet(n)` over elements `0` to `n - 1` with `find(x)` (the root of x's group, pointing every node on the way straight at the root), `union(a, b)` (merge two groups, hanging the smaller under the larger; `False` if already together), `size(x)` and a `groups` count.

**Example.** `DisjointSet(5)`: `union(0, 1)` -> `True`; `union(1, 0)` -> `False`; `size(1)` -> `2`; `groups` is `4`

**Big O.** O(alpha(N)) amortized per operation, effectively constant: union by size keeps the trees shallow, and path compression flattens every path that `find` walks.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-1)

In [ ]:
# 6.3.1 Union-find
class DisjointSet:
    def __init__(self, n):
        raise NotImplementedError

    def find(self, x):
        raise NotImplementedError

    def union(self, a, b):
        raise NotImplementedError

    def size(self, x):
        raise NotImplementedError

In [ ]:
def test_ex_6_3_1():
    ds = DisjointSet(5)
    assert ds.union(0, 1)
    assert not ds.union(1, 0)
    assert ds.size(1) == 2
    assert ds.groups == 4
    ds = DisjointSet(6)
    assert ds.union(0, 1)
    assert ds.union(2, 1)
    assert not ds.union(0, 2)
    assert ds.find(2) == ds.find(0)
    assert ds.size(2) == 3
    root = ds.find(0)
    assert ds.union(3, 0)
    assert ds.find(3) == root
    assert ds.size(3) == 4
    assert ds.groups == 3
    chain = DisjointSet(4)
    chain.parent = [0, 0, 1, 2]
    assert chain.find(3) == 0
    assert chain.parent == [0, 0, 0, 0]


test_ex_6_3_1()
print('6.3.1 passed')

### 6.3.2 · How many groups after the merges?  `Core`

**Task.** People `0` to `n - 1` start alone; each pair in `pairs` joins two people's groups. Write `group_count(n, pairs)` that returns how many groups remain, using your `DisjointSet` (6.3.1).

**Example.** `5, [[0, 1], [1, 2], [3, 4]]` -> `2`

**Big O.** O(N + M alpha(N)) for M pairs: one union per pair, and the set keeps the count.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-2)

In [ ]:
# 6.3.2 How many groups after the merges?
def group_count(n, pairs):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_2():
    assert group_count(5, [[0, 1], [1, 2], [3, 4]]) == 2
    assert group_count(3, []) == 3
    assert group_count(4, [[0, 1], [1, 0], [2, 3], [3, 1]]) == 1


test_ex_6_3_2()
print('6.3.2 passed')

### 6.3.3 · Cycle in an undirected graph  `Core`

**Task.** Write `has_cycle_undirected(n, edges)` for vertices `0` to `n - 1` and undirected `[u, v]` edges (no repeats, no self-loops): `True` when the graph has a cycle. Add edges to a `DisjointSet` one at a time; an edge inside one group closes a cycle.

**Example.** `4, [[0, 1], [1, 2], [2, 0], [2, 3]]` -> `True`; `4, [[0, 1], [1, 2], [2, 3]]` -> `False`

**Big O.** O(N + E alpha(N)): an edge whose ends are already in one group closes a cycle.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-3)

In [ ]:
# 6.3.3 Cycle in an undirected graph
def has_cycle_undirected(n, edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_3():
    assert has_cycle_undirected(4, [[0, 1], [1, 2], [2, 0], [2, 3]])
    assert not has_cycle_undirected(4, [[0, 1], [1, 2], [2, 3]])
    assert not has_cycle_undirected(3, [])
    assert has_cycle_undirected(5, [[3, 4], [0, 1], [1, 3], [4, 0]])


test_ex_6_3_3()
print('6.3.3 passed')

### 6.3.4 · Is the graph a tree?  `Core`

**Task.** Write `is_tree(n, edges)` for `n >= 1` vertices and undirected `[u, v]` edges: `True` when the graph is connected and has no cycle. Check that there are exactly `n - 1` edges and that none of them closes a cycle in a `DisjointSet`.

**Example.** `4, [[0, 1], [0, 2], [2, 3]]` -> `True`; `4, [[0, 1], [2, 3]]` -> `False`

**Big O.** O(N + E alpha(N)): a tree on N vertices has exactly N - 1 edges and no cycle, and those two facts together also force it to be connected.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-4)

In [ ]:
# 6.3.4 Is the graph a tree?
def is_tree(n, edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_4():
    assert is_tree(4, [[0, 1], [0, 2], [2, 3]])
    assert not is_tree(4, [[0, 1], [2, 3]])
    assert is_tree(1, [])
    assert not is_tree(4, [[0, 1], [1, 2], [2, 0]])
    assert not is_tree(3, [[0, 1], [1, 2], [2, 0]])


test_ex_6_3_4()
print('6.3.4 passed')

### 6.3.5 · Minimum spanning tree, edge by edge  `Practice`

**Task.** Write `kruskal_cost(n, edges)` for an undirected graph on `0` to `n - 1` with `[u, v, w]` edges: the total weight of a minimum spanning tree, or `-1` when the graph is not connected. Take edges cheapest first, keeping each one that joins two different groups of a `DisjointSet`.

**Example.** `4, [[0, 1, 4], [1, 2, 2], [0, 2, 5], [2, 3, 1], [1, 3, 7]]` -> `7`

**Big O.** O(E log E) for the sort, then O(E alpha(V)): the cheapest edge between two groups is always safe to take (the cut property).

**Note.** This is Kruskal's method; 6.3.6 grows the tree from one vertex instead.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-5)

In [ ]:
# 6.3.5 Minimum spanning tree, edge by edge
def kruskal_cost(n, edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_5():
    assert kruskal_cost(4, [[0, 1, 4], [1, 2, 2], [0, 2, 5], [2, 3, 1], [1, 3, 7]]) == 7
    assert kruskal_cost(1, []) == 0
    assert kruskal_cost(3, [[0, 1, 3]]) == -1
    assert kruskal_cost(3, [[0, 1, -2], [1, 2, 6], [0, 2, 6]]) == 4


test_ex_6_3_5()
print('6.3.5 passed')

### 6.3.6 · Minimum spanning tree, grown from one vertex  `Practice`

**Task.** Write `prim_cost(n, edges)` with the same input and answer as 6.3.5, but grow one tree from vertex `0`: keep a `heapq` of `(weight, vertex)` for edges leaving the tree, and add the cheapest one whose far end is not yet in the tree.

**Example.** `4, [[0, 1, 4], [1, 2, 2], [0, 2, 5], [2, 3, 1], [1, 3, 7]]` -> `7`

**Big O.** O(E log E) time, O(V + E) space: the heap always offers the cheapest edge leaving the tree built so far, and stale entries are skipped.

**Note.** This is Prim's method. It needs no union-find, and on dense graphs an array instead of a heap makes it O(V^2).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-6)

In [ ]:
# 6.3.6 Minimum spanning tree, grown from one vertex
def prim_cost(n, edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_6():
    assert prim_cost(4, [[0, 1, 4], [1, 2, 2], [0, 2, 5], [2, 3, 1], [1, 3, 7]]) == 7
    assert prim_cost(1, []) == 0
    assert prim_cost(3, [[0, 1, 3]]) == -1
    assert prim_cost(3, [[0, 1, -2], [1, 2, 6], [0, 2, 6]]) == 4
    edges = [[0, 1, 9], [0, 2, 3], [1, 2, 1], [1, 3, 8], [2, 3, 7], [3, 4, 2], [2, 4, 6]]
    assert prim_cost(5, edges) == 12


test_ex_6_3_6()
print('6.3.6 passed')

### 6.3.7 · The one extra edge  `Practice`

**Task.** `edges` connect vertices `0` to `n - 1`, where `n = len(edges)`: a tree plus one extra edge, so there is exactly one cycle. Write `redundant_edge(edges)` that returns the edge you can remove, in order of the list, to leave a tree: the first one that joins two vertices already in one group.

**Example.** `[[0, 1], [1, 2], [2, 3], [1, 3], [3, 4]]` -> `[1, 3]`

**Big O.** O(N alpha(N)): add edges in order; the first one whose ends are already joined is the extra one.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-7)

In [ ]:
# 6.3.7 The one extra edge
def redundant_edge(edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_7():
    assert redundant_edge([[0, 1], [1, 2], [2, 3], [1, 3], [3, 4]]) == [1, 3]
    assert redundant_edge([[0, 1], [1, 2], [0, 2]]) == [0, 2]
    assert redundant_edge([[2, 3], [0, 1], [1, 3], [0, 2]]) == [0, 2]


test_ex_6_3_7()
print('6.3.7 passed')

### 6.3.8 · When is everyone connected?  `Practice`

**Task.** `logs` holds `[time, a, b]` entries, in any order, each saying people `a` and `b` became friends at `time`; friendship joins groups. Write `earliest_connected(n, logs)` for `n >= 2` people that returns the earliest time everyone is in one group, or `-1`. Sort by time and merge until `groups` reaches one.

**Example.** `4, [[7, 2, 3], [2, 0, 1], [5, 1, 2], [9, 0, 3]]` -> `7`

**Big O.** O(M log M) for sorting M log entries, then O(M alpha(N)): the group count falls to one at exactly the answer.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-3-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-3-8)

In [ ]:
# 6.3.8 When is everyone connected?
def earliest_connected(n, logs):
    raise NotImplementedError

In [ ]:
def test_ex_6_3_8():
    assert earliest_connected(4, [[7, 2, 3], [2, 0, 1], [5, 1, 2], [9, 0, 3]]) == 7
    assert earliest_connected(3, [[1, 0, 1]]) == -1
    assert earliest_connected(2, [[4, 1, 0], [3, 0, 1]]) == 3
    assert earliest_connected(3, [[5, 0, 1], [6, 1, 0], [8, 2, 1]]) == 8


test_ex_6_3_8()
print('6.3.8 passed')